# 01 — Probability Foundations
**References:** DeGroot & Schervish (2012) Ch. 1–2 · Casella & Berger (2002) Ch. 1

## Narrative thread
```
Sample space -> Sigma-algebra -> Probability measure -> Conditional probability -> Independence
```

## 1. Kolmogorov's axioms

A **probability space** is a triple $(\Omega, \mathcal{F}, P)$ where:

- $\Omega$ — sample space: the set of all possible outcomes
- $\mathcal{F}$ — sigma-algebra: a collection of subsets of $\Omega$ closed under countable unions, intersections, and complements
- $P : \mathcal{F} \to [0,1]$ — probability measure satisfying Kolmogorov's three axioms:

$$\text{(K1) } P(A) \geq 0 \quad \forall A \in \mathcal{F}$$

$$\text{(K2) } P(\Omega) = 1$$

$$\text{(K3) If } A_1, A_2, \ldots \text{ are mutually disjoint, then } P\!\left(\bigcup_{i=1}^{\infty} A_i\right) = \sum_{i=1}^{\infty} P(A_i)$$

**Derived properties** (all follow from K1–K3):

| Property | Formula |
|---|---|
| Complement | $P(A^c) = 1 - P(A)$ |
| Empty set | $P(\emptyset) = 0$ |
| Monotonicity | $A \subseteq B \Rightarrow P(A) \leq P(B)$ |
| Union | $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ |
| Boole's inequality | $P\!\left(\bigcup_i A_i\right) \leq \sum_i P(A_i)$ |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from itertools import combinations

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── Verify axioms via Monte Carlo simulation ─────────────────────────────
# Experiment: roll a fair die, Omega = {1,2,3,4,5,6}
n = 100_000
rolls = np.random.randint(1, 7, size=n)

A = rolls <= 3          # event A: {1,2,3}
B = rolls >= 4          # event B: {4,5,6}  -- A and B are disjoint
C = rolls % 2 == 0     # event C: {2,4,6}

print('=== Kolmogorov axioms — Monte Carlo verification ===')
print(f'  K1: P(A) = {A.mean():.4f}  >= 0  ✓')
print(f'  K2: P(Omega) = {np.ones(n).mean():.4f} = 1  ✓')
print(f'  K3: P(A∪B) = {(A|B).mean():.4f}  =  P(A)+P(B) = {A.mean()+B.mean():.4f}  ✓  (A,B disjoint)')
print()
print('=== Derived properties ===')
print(f'  Complement:   P(A^c) = {(~A).mean():.4f}  =  1-P(A) = {1-A.mean():.4f}')
print(f'  Union:        P(A∪C) = {(A|C).mean():.4f}  =  P(A)+P(C)-P(A∩C) = {A.mean()+C.mean()-(A&C).mean():.4f}')
print(f"  Boole's ineq: P(A∪C) = {(A|C).mean():.4f}  <=  P(A)+P(C) = {A.mean()+C.mean():.4f}")

## 2. Conditional probability and the multiplication rule

The **conditional probability** of $A$ given $B$ (with $P(B)>0$) is:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

**Multiplication rule:**
$$P(A \cap B) = P(A \mid B)\,P(B) = P(B \mid A)\,P(A)$$

**Law of Total Probability:** if $\{B_1, \ldots, B_k\}$ is a partition of $\Omega$,
$$P(A) = \sum_{i=1}^k P(A \mid B_i)\,P(B_i)$$

**Bayes' Theorem:**
$$P(B_i \mid A) = \frac{P(A \mid B_i)\,P(B_i)}{\sum_{j=1}^k P(A \mid B_j)\,P(B_j)}$$

> **Intuition:** Bayes' theorem inverts the direction of conditioning. It is the foundation of Bayesian statistical inference (notebook 11).

In [ ]:
# ── Medical test example — Bayes' theorem ────────────────────────────────
# Disease prevalence, sensitivity, specificity
prevalence   = 0.01   # P(disease)
sensitivity  = 0.99   # P(test+ | disease)  -- true positive rate
specificity  = 0.95   # P(test- | no disease)
false_pos    = 1 - specificity  # P(test+ | no disease)

# P(test+)  by law of total probability
p_test_pos = sensitivity * prevalence + false_pos * (1 - prevalence)

# P(disease | test+)  by Bayes
p_disease_given_pos = (sensitivity * prevalence) / p_test_pos

print('=== Medical test: Bayes theorem ===')
print(f'  P(disease)       = {prevalence:.2f}')
print(f'  P(test+ | dis.)  = {sensitivity:.2f}  (sensitivity)')
print(f'  P(test+ | no dis)= {false_pos:.2f}  (false positive rate)')
print(f'  P(test+)         = {p_test_pos:.4f}  (law of total probability)')
print(f'  P(dis | test+)   = {p_disease_given_pos:.4f}  <-- posterior  (Bayes theorem)')
print()
print('  Interpretation: even with 99% sensitivity, a positive test in a rare')
print(f'  disease means only {p_disease_given_pos*100:.1f}% chance of actually having it.')
print('  Prior prevalence dominates when the disease is rare.')

# ── Visualize: posterior vs prevalence ───────────────────────────────────
prevalences = np.linspace(0.001, 0.50, 300)
posteriors  = (sensitivity * prevalences) / (sensitivity * prevalences + false_pos * (1 - prevalences))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(prevalences * 100, posteriors * 100, color='#4361ee', lw=2.5)
ax.axvline(prevalence * 100, color='#f72585', lw=1.5, linestyle='--', label=f'prevalence={prevalence*100:.0f}%')
ax.axhline(p_disease_given_pos * 100, color='#06d6a0', lw=1.5, linestyle='--',
           label=f'posterior={p_disease_given_pos*100:.1f}%')
ax.set_xlabel('Disease prevalence (%)')
ax.set_ylabel('P(disease | test+) %')
ax.set_title('Bayes theorem: posterior probability vs prior prevalence\n'
             'sensitivity=99%, specificity=95%')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Independence

Events $A$ and $B$ are **independent** if and only if:
$$P(A \cap B) = P(A)\,P(B)$$

Equivalently (when $P(B)>0$): $P(A \mid B) = P(A)$ — knowing $B$ gives no information about $A$.

**Mutual independence** of $\{A_1, \ldots, A_n\}$ requires:
$$P\!\left(\bigcap_{i \in S} A_i\right) = \prod_{i \in S} P(A_i) \quad \text{for every subset } S \subseteq \{1,\ldots,n\}$$

> **Warning:** Pairwise independence $\not\Rightarrow$ mutual independence (Bernstein's example).

In [ ]:
# ── Bernstein's counterexample: pairwise but NOT mutually independent ─────
# Toss a fair coin 3 times. Define:
#   A = {HHT, HTH, HHH, THH} -- at least 2 heads (we'll use a simpler version)
# Classic Bernstein: 4 equally likely outcomes {1,2,3,4}
#   A = {1,4}, B = {2,4}, C = {3,4}

n = 200_000
omega = np.random.choice([1, 2, 3, 4], size=n)  # uniform on {1,2,3,4}

A = (omega == 1) | (omega == 4)  # P(A) = 1/2
B = (omega == 2) | (omega == 4)  # P(B) = 1/2
C = (omega == 3) | (omega == 4)  # P(C) = 1/2

pA, pB, pC = A.mean(), B.mean(), C.mean()
pAB, pAC, pBC = (A&B).mean(), (A&C).mean(), (B&C).mean()
pABC = (A&B&C).mean()

print('=== Bernstein counterexample: pairwise ≠ mutual independence ===')
print(f'  P(A)={pA:.3f}  P(B)={pB:.3f}  P(C)={pC:.3f}')
print()
print('  Pairwise independence check:')
print(f'    P(A∩B)={pAB:.3f}  vs  P(A)·P(B)={pA*pB:.3f}  -> {"INDEPENDENT" if abs(pAB-pA*pB)<0.01 else "DEPENDENT"}')
print(f'    P(A∩C)={pAC:.3f}  vs  P(A)·P(C)={pA*pC:.3f}  -> {"INDEPENDENT" if abs(pAC-pA*pC)<0.01 else "DEPENDENT"}')
print(f'    P(B∩C)={pBC:.3f}  vs  P(B)·P(C)={pB*pC:.3f}  -> {"INDEPENDENT" if abs(pBC-pB*pC)<0.01 else "DEPENDENT"}')
print()
print('  Mutual independence check:')
print(f'    P(A∩B∩C)={pABC:.3f}  vs  P(A)·P(B)·P(C)={pA*pB*pC:.3f}  -> {"INDEPENDENT" if abs(pABC-pA*pB*pC)<0.01 else "NOT mutually independent!"}')

## 4. Counting and combinatorics

When $\Omega$ is finite and all outcomes are equally likely, $P(A) = |A|/|\Omega|$.

| Tool | Formula | Use |
|---|---|---|
| Permutations (ordered, no replacement) | $P(n,r) = \frac{n!}{(n-r)!}$ | Ordered selection |
| Combinations (unordered, no replacement) | $\binom{n}{r} = \frac{n!}{r!(n-r)!}$ | Subset selection |
| Multinomial | $\binom{n}{k_1,\ldots,k_m} = \frac{n!}{k_1!\cdots k_m!}$ | Multiclass allocation |

**Binomial theorem:**
$$(a+b)^n = \sum_{k=0}^n \binom{n}{k} a^k b^{n-k}$$

Setting $a=p$, $b=1-p$ gives the binomial probabilities: $P(X=k) = \binom{n}{k}p^k(1-p)^{n-k}$.

In [ ]:
from math import comb, factorial, perm

# ── Birthday problem ─────────────────────────────────────────────────────
# P(at least 2 people share a birthday in a room of n) = 1 - P(all distinct)
def p_shared_birthday(n):
    """Probability that at least 2 of n people share a birthday."""
    p_all_distinct = 1.0
    for k in range(n):
        p_all_distinct *= (365 - k) / 365
    return 1 - p_all_distinct

ns = np.arange(1, 80)
probs = [p_shared_birthday(n) for n in ns]

n50 = next(n for n, p in zip(ns, probs) if p >= 0.50)
n99 = next(n for n, p in zip(ns, probs) if p >= 0.99)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ns, probs, color='#4361ee', lw=2.5)
ax.axhline(0.5, color='#f72585', lw=1.5, linestyle='--', label=f'50% at n={n50}')
ax.axhline(0.99, color='#06d6a0', lw=1.5, linestyle='--', label=f'99% at n={n99}')
ax.set_xlabel('Number of people in room')
ax.set_ylabel('P(at least 2 share birthday)')
ax.set_title('Birthday problem — equally likely outcomes + combinatorics\n'
             'Result surprises most: only 23 people needed for >50% chance')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nOnly {n50} people needed for >50% chance of shared birthday.')
print(f'Need {n99} people for >99% chance.')

## 5. Key takeaways

| Concept | What to remember |
|---|---|
| Kolmogorov axioms | All of probability theory follows from 3 axioms |
| Bayes' theorem | Inverts conditioning; core of Bayesian inference (nb. 11) |
| Independence | $P(A\cap B) = P(A)P(B)$; pairwise $\neq$ mutual |
| Combinatorics | Equally-likely sample spaces reduce probability to counting |

**Next:** notebook 02 formalizes random variables — functions that map $\Omega$ to $\mathbb{R}$ — and their distributions.